# Gray-Scott 128x128: 10k snapshots

This notebook generates a higher-resolution Gray-Scott dataset for the project: `10_000` snapshots, `128x128`, one `v` concentration channel, quality filtering, sample previews, and consecutive trajectory previews.

## 1. Runtime check

Use a GPU runtime. A100 is ideal; L4/T4 also work but may be slower.

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Clone repository

The repository is public, so no GitHub token is needed.

In [ ]:
import subprocess

REPO_URL = "https://github.com/SadreevAmir/DL_final_project.git"
REPO_DIR = "gray_scott_dataset_repo"

subprocess.run(["rm", "-rf", REPO_DIR], check=True)
subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
%cd {REPO_DIR}
!pip install -q -r requirements.txt

## 3. Generate dataset

This preset is intended to finish in roughly half an hour on A100. It downloads every `.npz` chunk and every preview image as soon as they are saved.

In [ ]:
from pathlib import Path
from google.colab import files
from IPython.display import display
from PIL import Image

from grayscott_dataset import GrayScottConfig, generate_dataset

output_dir = Path("data/grayscott_128_10k")

config = GrayScottConfig(
    output_dir=str(output_dir),
    total_images=10_000,
    grid_size=128,
    num_trajectories=500,
    max_trajectories=2_000,
    snapshots_per_trajectory=20,
    burn_in_steps=3_000,
    save_interval=25,
    chunk_size=1_000,
    sim_batch_size=250,
    dt=1.0,
    solver_substeps=2,
    channels="v",
    dtype="float16",
    compress=False,
    param_mode="mixed",
    seed=42,
    device="cuda" if torch.cuda.is_available() else "cpu",
    num_threads=12,
    save_previews=True,
    preview_every_chunks=1,
    preview_count=32,
    save_sequence_previews=True,
    sequence_preview_count=16,
    min_image_std=0.025,
    min_image_range=0.15,
)

def show_and_download_preview(path: Path):
    print(f"Preview: {path}")
    display(Image.open(path))
    files.download(str(path))

def download_chunk(path: Path):
    print(f"Chunk: {path}")
    files.download(str(path))

paths = generate_dataset(
    config,
    on_chunk_saved=download_chunk,
    on_preview_saved=show_and_download_preview,
)

print(f"Saved {len(paths)} chunks")
files.download(str(output_dir / "manifest.json"))

## 4. Inspect generated data

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

manifest = json.loads((output_dir / "manifest.json").read_text())
print(json.dumps({
    "total_images": manifest["total_images"],
    "accepted_images": manifest["accepted_images"],
    "rejected_images": manifest["rejected_images"],
    "simulated_trajectories": manifest["simulated_trajectories"],
    "elapsed_seconds": manifest["elapsed_seconds"],
    "shape": manifest["shape"],
}, indent=2))

chunk = np.load(paths[0])
images = chunk["images"]
print("First chunk images:", images.shape, images.dtype, float(images.min()), float(images.max()))

fig, axes = plt.subplots(4, 8, figsize=(12, 6))
for ax, img in zip(axes.ravel(), images[:32]):
    ax.imshow(img[0], cmap="magma", vmin=0, vmax=1)
    ax.axis("off")
plt.tight_layout()

## 5. Optional: zip full folder

Chunks are already downloaded individually. This cell creates one archive as a backup.

In [ ]:
archive_path = "grayscott_128_10k_dataset.zip"
!zip -q -r {archive_path} {output_dir}
files.download(archive_path)